# Geodemographics

This notebook contains the full workflow for producing a geodemographic classification from scratch in using k-means clustering. 


* **Data processing**
    * Transform variables
    * Standardise variables.
    * Perform correlation & variance analysis to identify potentially redundant variables.

* **Clustering:**
    * Determine optimal number of clusters using Clustergrams.
    * Apply K-Means clustering to classify areas based on selected variables.
    * Perform top-down hierarchical clustering to divide clusters into subgroups.
    
* **Analytical Techniques:**
    * Use UMAP (Uniform Manifold Approximation and Projection) to visualise high-dimensional embeddings in 2D.


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.cluster import KMeans
import umap.umap_ as umap
from clustergram import Clustergram
import math

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from matplotlib.colors import ListedColormap
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# set a  random seed for reproducibility
random_seed = 42

In [ ]:
# | echo: false

# Set global font sizes (affects titles, labels, ticks, etc.)
plt.rcParams.update(
    {
        "axes.titlesize": 7,  # subplot titles
        "axes.labelsize": 8,  # x/y labels
        "xtick.labelsize": 6,  # x tick labels
        "ytick.labelsize": 6,  # y tick labels
        "legend.fontsize": 6,  # legend
        "figure.figsize": (8, 6),  # default figure size
    }
)

# Data processing

In [ ]:
df = gpd.read_parquet(
    "/data/uscuni-restricted/geodem/05_data/_relative_merged_census_2021_gender.parquet"
)
variable_df = df.drop(
    columns=[
        "celkem_CIZ",
        "Obyvatelstvo - ekon. aktivita: zaměstnaní - muži",
        "Obyvatelstvo - ekon. aktivita: zaměstnaní - ženy",
    ]
)
# remove variables
variable_df = variable_df.drop(
    columns=[
        "Obyvatelstvo - státní občanství: Česká republika - muži",
        "Obyvatelstvo - státní občanství: Česká republika - ženy",
        "Obyvatelstvo - státní občanství: Slovenská republika - muži",
        "Obyvatelstvo - státní občanství: Slovenská  republika - ženy",
        "Obyvatelstvo - státní občanství: země EU mimo ČR - muži",
        "Obyvatelstvo - státní občanství: země EU mimo ČR - ženy",
        "Obyvatelstvo - bez státního občanství - muži",
        "Obyvatelstvo - bez státního občanství - ženy",
        "Obyvatelstvo - mimo pracovní sílu - muži",
        "Obyvatelstvo - mimo pracovní sílu - ženy",
    ]
)

In [ ]:
# load csv file with information
var_lookup = pd.read_csv(
    "/data/uscuni-restricted/geodem/00_help_files/_var_lookup_gender.csv"
).set_index("Unnamed: 0")

In [ ]:
# rename to english
variable_df = variable_df.rename(
    columns=(dict(zip(var_lookup["full_names_CZ"], var_lookup["short_name_ENG"])))
)

## Standardization

Before applying clustering algorithms, it is important to transform and standardise the data to ensure that all variables contribute equally to the analysis.

The function below applies two transformations to a dataframe (applied column-wise):

1. **Inverse Hyperbolic Sine (IHS) transform**  
2. **Min–Max scaling to [0, 1]**

### Mathematical definitions

**Inverse hyperbolic sine (IHS, a.k.a. arcsinh):**  
- Similar to a log transform but works with zero and negative values.  
- Helps stabilise variance and make skewed distributions more normal-like.  

$$
\mathrm{arcsinh}(x) = \ln\!\big(x + \sqrt{x^{2}+1}\big)
$$

**Properties:**
- For large \(|x|\):  
  $$
  \mathrm{arcsinh}(x) \approx \ln(2|x|)
  $$
  (behaves like log).  

- Near \(0\):  
  $$
  \mathrm{arcsinh}(x) \approx x
  $$

 

**Min–Max scaling (applied per column after IHS):**

$$
x' = \frac{x - \min(x)}{\max(x) - \min(x)}
$$

- Rescales all values into the range \([0, 1]\).  
- Useful for comparing variables with different units/scales.

In [ ]:
variables = variable_df.columns.drop("geometry")


def standardise_data(df):
    """
    Apply inverse hyperbolic sine transform to handle non-normality,
    then apply Z-score scaling (Standardization).
    """
    df_transformed = np.arcsinh(df)
    # We use StandardScaler to ensure mean=0 and variance=1
    #   scaler = MinMaxScaler()

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_transformed)
    return pd.DataFrame(scaled_data, columns=df.columns, index=df.index)


# Apply to your dataframe
scaled_variable_df = variable_df.copy()
scaled_variable_df[variables] = standardise_data(variable_df[variables])

In [ ]:
# 1. Data
df_before = variable_df.drop(columns="geometry").copy()
df_after = scaled_variable_df.drop(columns="geometry").copy()

# 2. Layout Settings
num_vars = len(df_before.columns)
ncols = 6
nrows = int(np.ceil(num_vars / ncols))

# 3. Create Plot
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(df_before.columns):
    ax = axes[i]

    # Plot Before (Red) - Using twinx to handle different scales
    ax.set_title(f"{col}", fontsize=10, fontweight="bold")
    sns.kdeplot(df_before[col], ax=ax, color="red", fill=True, label="Before (Orig)")
    ax.tick_params(axis="x", colors="red")
    ax.set_xlabel("Original Scale", color="red")

    # Plot After (Blue) on a second X-axis
    ax2 = ax.twiny()
    sns.kdeplot(
        df_after[col], ax=ax2, color="blue", fill=True, label="After (Transformed)"
    )
    ax2.tick_params(axis="x", colors="blue")
    ax2.set_xlabel("Transformed Scale", color="blue")

    # Clean up formatting
    if i == 0:
        lines, labels = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines + lines2, labels + labels2, loc="upper right")

# Remove empty subplots if 170 is not divisible by 3
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## PCA for dimensionality reduction


In [ ]:
# Determine the number of components using Kaiser's Criterion (Harsh Threshold)
pca_initial = PCA().fit(scaled_variable_df.drop(columns="geometry"))
eigenvalues = pca_initial.explained_variance_
N = np.sum(eigenvalues >= 1)

print(f"Using Kaiser's Criterion: N = {N} components.")

In [ ]:
# 1. Prepare data and limit to first 50
limit = 50
factors = np.arange(1, len(eigenvalues) + 1)

# Slice data to ensure we don't plot beyond the available components or the limit
plot_range = slice(0, min(len(eigenvalues), limit))

fig, ax1 = plt.subplots(figsize=(15, 8))

# --- Left Axis: Eigenvalues (Scree Plot) ---
ax1.set_xlabel("Factors (Principal Components)")
ax1.set_ylabel("Eigenvalue", color="tab:blue")
ax1.plot(
    factors[plot_range],
    eigenvalues[plot_range],
    marker="o",
    color="tab:blue",
    label="Eigenvalue",
)
ax1.axhline(1, color="black", linestyle="--", alpha=0.6, label="Kaiser Threshold")
ax1.tick_params(axis="y", labelcolor="tab:blue")

# --- Right Axis: Growing Ratio (Cumulative Variance) ---
ax2 = ax1.twinx()
cumulative_variance = np.cumsum(pca_initial.explained_variance_ratio_)
ax2.set_ylabel("Cumulative Explained Variance", color="red")
ax2.plot(
    factors[plot_range],
    cumulative_variance[plot_range],
    marker="s",
    color="red",
    linewidth=2,
    label="Growing Ratio",
)
ax2.tick_params(axis="y", labelcolor="red")

# --- Aesthetics ---
plt.title(f"PCA Analysis (First {limit} Components)")
ax1.set_xlim(0.5, limit + 0.5)  # Focuses the x-axis on your limit
ax1.grid(True, which="both", linestyle="--", alpha=0.5)

# Combined Legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.1),
    ncol=3,
)

fig.tight_layout()
plt.show()

In [ ]:
# 1. Calculate the cumulative variance array
cumulative_variance = np.cumsum(pca_initial.explained_variance_ratio_)

# 2. Extract values for the 32nd component (index 31)
n_idx = 31
target_eigenvalue = eigenvalues[n_idx]
retained_variance = cumulative_variance[n_idx] * 100

print("--- PCA Results for 32 Components ---")
print(f"Eigenvalue of the 32nd component: {target_eigenvalue:.4f}")
print(f"Total variance retained: {retained_variance:.2f}%")

In [ ]:
# Initialize PCA with your chosen 32 components
pca_32 = PCA(n_components=32)

# Fit and Transform the data
pc_data = pca_32.fit_transform(scaled_variable_df.drop(columns="geometry"))

# Create a new DataFrame for the Principal Components
pc_columns = [f"PC{i + 1}" for i in range(32)]
df_pca = pd.DataFrame(data=pc_data, columns=pc_columns, index=scaled_variable_df.index)

# Re-attach the geometry column so you can still plot maps
pca_df = pd.concat([scaled_variable_df[["geometry"]], df_pca], axis=1)

### Clustering

## Choosing number of clusters

In [ ]:
# Since k-means is sensitive to initialization, `n_init` determines the number of
# times the algorithm runs with different centroid seeds. The final result is the
# best outcome based on inertia/WCSS (within-cluster sum of squares).

n_init = 100  # Use a low value for quick testing, increase (~100) for final results
cgram = Clustergram(
    range(1, 20), n_init=n_init, random_state=random_seed, verbose=False
)
# Fit model to data
cgram.fit(pca_df.drop(columns="geometry"))
# Generate plot
cgram.plot()

## Apply K-Means Clustering

In [ ]:
# 1. Set parameters
num_clusters = 7
n_init = 1000
sort_variable = "pop_edu_university_f"

# 2. Initialize and Fit K-means
# We work on a copy to keep the original data clean
supergrouped_variable_df = variable_df.copy()
features = supergrouped_variable_df.drop(columns="geometry", errors="ignore")

kmeans_model = KMeans(
    n_clusters=num_clusters, init="random", n_init=n_init, random_state=random_seed
)
# Assign initial numeric clusters (these will be random)
supergrouped_variable_df["cluster_temp"] = kmeans_model.fit_predict(features)

# 3. Reorder Logic
# Calculate the mean of the sort_variable for each temporary cluster
ordered_indices = (
    supergrouped_variable_df.groupby("cluster_temp")[sort_variable]
    .mean()
    .sort_values()
    .index
)

# Create a mapping from the random K-means ID to the sorted Numeric ID (0 to 9)
reorder_map = {
    old_label: new_label for new_label, old_label in enumerate(ordered_indices)
}
supergrouped_variable_df["cluster_numeric"] = supergrouped_variable_df[
    "cluster_temp"
].map(reorder_map)

# 4. Map Numeric IDs to Letters (0 -> A, 1 -> B, etc.)
label_map = {i: chr(65 + i) for i in range(num_clusters)}
supergrouped_variable_df["cluster"] = supergrouped_variable_df["cluster_numeric"].map(
    label_map
)

# 5. Cleanup
# Remove the temporary columns used for calculation
supergrouped_variable_df = supergrouped_variable_df.drop(
    columns=["cluster_temp", "cluster_numeric"]
)

# View the result - 'A' is now your 'Lowest' group and 'J' is your 'Highest'
print(supergrouped_variable_df["cluster"].value_counts().sort_index())

In [ ]:
# 1. Get the unique subclusters actually present in your data
colors_super_groups = {
    # A: Blue (Base: #1f77b4)
    "A2": "#15527c",
    "A1": "#8c564b",
    "A": "#1f77b4",
    # B: Orange (Base: #ff7f0e)
    "B1": "#17becf",
    "C": "#ff7f0e",
    "B2": "#b3590a",
    # C: Green (Base: #2ca02c)
    "C1": "#98df8a",
    "B": "#2ca02c",
    "C2": "#1e6e1e",
    # D: Purple/Pink (Base: #9467bd)
    "F": "#c5b0d5",
    "D": "#9467bd",
    "D2": "#674783",
    # E: Pink/Magenta (Base: #e377c2)
    "E1": "#d62728",
    "E": "#e377c2",
    "F1": "#000000",
    "E3": "khaki",
    "G": "khaki",
}


# 1. Get the unique clusters present in the data and sort them
# Sorting ensures the Colormap aligns with the alphabetical legend (A, A0, A1...)
present_categories = sorted(supergrouped_variable_df["cluster"].unique())

shades_supergroups = [colors_super_groups[cat] for cat in present_categories]

# 3. Create the colormap
my_custom_cmap = ListedColormap(shades_supergroups)

# 4. Plot
# We sort the dataframe by cluster so the plotting engine matches the sorted cmap
supergrouped_variable_df.sort_values("cluster").plot(
    column="cluster",
    categorical=True,
    cmap=my_custom_cmap,
    legend=True,
)

In [ ]:
supergrouped_variable_df[["cluster", "geometry"]].assign(
    geometry=lambda df: df.geometry.simplify(25, preserve_topology=True)
).explore(
    "cluster", categorical=True, legend=True, cmap=my_custom_cmap, prefer_canvas=True
)

## UMAP Visualisation

We can use UMAP (Uniform Manifold Approximation and Projection) [@umap] to visualise the high-dimensional Census data in 2D. UMAP is a dimensionality reduction technique that preserves both local and global structure in the data, making it well-suited for visualising complex datasets like Census data.

In [ ]:
# Fit UMAP
# Apply UMAP to reduce 64 dimensions to 2D
reducer = umap.UMAP(
    n_neighbors=30,  # Numbers of neighbours
    min_dist=0.0,  # Allow points to be closer together
    n_components=7,  # Reduce to 2D for visualsation
    random_state=42,  # For reproducible results
    metric="cosine",  # Cosine similarity
    init="random",  # Use random initialisation
    n_epochs=500,  # More epochs for better convergence
    spread=1.0,  # Controls how tightly points are packed
    verbose=False,  # Show progress
)

embedding = reducer.fit_transform(supergrouped_variable_df[variables].values)

umap_results = pd.DataFrame(
    {
        "UMAP1": embedding[:, 0],
        "UMAP2": embedding[:, 1],
        "UMAP3": embedding[:, 2],
        "UMAP4": embedding[:, 3],
        "UMAP5": embedding[:, 4],
        "UMAP6": embedding[:, 5],
        "UMAP7": embedding[:, 6],
        "Cluster": supergrouped_variable_df["cluster"].values,
    }
)
# Create interactive UMAP scatter plot
fig_interactive = px.scatter(
    umap_results,
    x="UMAP1",
    y="UMAP2",
    color="Cluster",
    category_orders={"Cluster": sorted(umap_results["Cluster"].unique())},  #
    color_discrete_map=colors_super_groups,
)

# Style tweaks
fig_interactive.update_traces(marker=dict(size=3, opacity=0.7))
fig_interactive.update_layout(
    title="UMAP Projection of Clusters",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    legend_title="Cluster",
)

fig_interactive.update_layout(width=700, height=600)
fig_interactive.show(config={"responsive": False})

In [ ]:
# get total population
pop_size = pd.read_csv(
    "/data/uscuni-restricted/geodem/04_spatial/total.csv",
    dtype={"nadzsjd": str},
    index_col=0,
)

# basic statistics of each cluster, number (perc of OAs) in each cluster and population

# number of OAs in each cluster
cluster_counts = supergrouped_variable_df["cluster"].value_counts().sort_index()
# percentage of OAs in each cluster
cluster_perc = (cluster_counts / cluster_counts.sum() * 100).round(2)

# join pop_size to supergrouped_variable_df on index
supergrouped_variable_df_withpop = supergrouped_variable_df.join(pop_size, how="left")

# pop in each cluster
cluster_pop = supergrouped_variable_df_withpop.groupby("cluster")[
    "Obyvatelstvo celkem"
].sum()
# percentage of pop in each cluster
cluster_pop_perc = (cluster_pop / cluster_pop.sum() * 100).round(2)

# combine into a dataframe
cluster_summary = pd.DataFrame(
    {
        "Number of OAs": cluster_counts,
        "Percentage of OAs": cluster_perc,
        "Population": cluster_pop,
        "Percentage of Population": cluster_pop_perc,
    }
)
cluster_summary

In [ ]:
cluster_means = (
    supergrouped_variable_df.drop(columns="geometry").groupby("cluster").mean()
)
global_means = supergrouped_variable_df.drop(columns=["geometry", "cluster"]).mean()
# --- Calculate percentage difference ---
pct_diff = (cluster_means / global_means) * 100
# drop columns with nan
pct_diff = pct_diff.dropna(axis=1, how="any")


# Transpose for easier plotting (rows: encodings, columns: clusters)
pct_display_df = pct_diff.T
# build human names for hover
feature_names = pct_display_df.index.to_numpy(dtype=object)
customdata_pct = np.tile(feature_names.reshape(-1, 1), (1, pct_display_df.shape[1]))

# --- Heatmap (percentage difference) ---
fig_pct = px.imshow(
    pct_display_df,
    color_continuous_scale="RdYlGn",
    origin="lower",
    aspect="auto",
    labels=dict(x="Cluster", y="Feature (encoding)", color="% of global mean"),
    zmin=0,
    zmax=200,
)

# attach customdata and set hover
fig_pct.data[0].customdata = customdata_pct
fig_pct.update_traces(
    hovertemplate="Cluster: %{x}<br>Encoding: %{y}<br>Name: %{customdata}<br>% of Global Mean: %{z:.1f}%<extra></extra>",
    zmid=100,  # centre colours on 100%
)

fig_pct.update_layout(
    title="Cluster Profiles (% of Global Mean)",
    xaxis_title="Cluster",
    yaxis_title="Feature (encoding)",
    height=2500,
)

fig_pct.show()

In [ ]:
# 1. Setup dimensions
categories = pct_diff.columns.tolist()
clusters = pct_diff.index.tolist()
num_clusters = len(clusters)

cols = 2
rows = (num_clusters + 1) // cols

fig = make_subplots(
    rows=rows,
    cols=cols,
    specs=[[{"type": "polar"}] * cols] * rows,
    subplot_titles=[f"Cluster {c}" for c in clusters],
)

for i, cluster in enumerate(clusters):
    row = (i // cols) + 1
    col = (i % cols) + 1

    # Select color based on cluster ID (defaults to grey if ID not in dict)
    cluster_color = colors_super_groups.get(str(cluster), "#7f7f7f")

    r_values = list(pct_diff.loc[cluster].values)
    r_plot = r_values + [r_values[0]]
    theta_plot = categories + [categories[0]]

    fig.add_trace(
        go.Scatterpolar(
            r=r_plot,
            theta=theta_plot,
            fill="toself",
            name=f"Cluster {cluster}",
            # Use your custom colors
            line=dict(color=cluster_color),
            fillcolor=cluster_color,
            opacity=0.6,
            # Tooltip logic
            mode="lines+markers",
            marker=dict(size=0, opacity=0),
            hoveron="points",
            hovertemplate="<b>Feature:</b> %{theta}<br><b>Value:</b> %{r:.1f}%<extra></extra>",
        ),
        row=row,
        col=col,
    )

# 2. Layout and Clean Axis
fig.update_layout(
    title_text="Cluster Profiles (% of Global Mean)",
    height=400 * rows,
    showlegend=False,
    template="plotly_white",
    hovermode="closest",
)

fig.update_polars(
    radialaxis=dict(visible=True, range=[0, 200], ticksuffix="%"),
    angularaxis=dict(
        showticklabels=False,  # Hides the names from the chart axes
        ticks="",
        showline=True,
        linecolor="lightgrey",
    ),
)

fig.show()

# Hierarchical Subclustering

We often want to perform a finer level of clustering to capture more detailed patterns in the data.
For OAC the top level "supergroup" clusters are split further into groups and subgroups by applying the above process iteratively. This process is referred to as top down clustering. This has the advantage of allowing more clusters to be created without needing to consider all clusters at once. It also allows for more interpretable clusters as the subclusters are nested within the broader supergroup clusters.

## Selecting the Number of Subclusters
We can use clustergrams again to select the number of subclusters for each supergroup.
We create clustergrams for each supergroup and select the number of subclusters based on the same principles as before.

In [ ]:
def create_subcluster_clustergrams(output_df, n_init=1, cols=2):
    # 1. Define global limits
    global_min, global_max = -50, 150
    cluster_labels = np.sort(output_df["cluster"].unique())
    num_plots = len(cluster_labels)

    # 2. Calculate grid dimensions (rows x cols)
    rows = math.ceil(num_plots / cols)

    # 3. Create the figure
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
    axes = axes.flatten()  # Flatten in case of multi-row grid

    for i, cluster_label in enumerate(cluster_labels):
        cluster_df = output_df.query("cluster == @cluster_label").drop(
            columns="cluster"
        )

        if cluster_df.empty:
            continue

        # Generate clustergram
        # Note: Ensure 'Clustergram' and 'random_seed' are defined in your environment
        cgram_sub = Clustergram(
            range(1, 10), n_init=n_init, random_state=random_seed, verbose=False
        )
        cgram_sub.fit(cluster_df)

        # 4. Plot on the specific subplot axis
        cgram_sub.plot(ax=axes[i])

        # 5. Apply formatting to that specific axis
        axes[i].set_ylim(global_min, global_max)
        axes[i].set_title(f"Clustergram: Cluster {cluster_label}")

    # 6. Hide any unused empty subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()


# Example usage (cols=3 will put 3 plots per row)
create_subcluster_clustergrams(
    supergrouped_variable_df.drop(columns="geometry"), n_init=10, cols=3
)

In [ ]:
supergrouped_variable_df.sort_values("cluster").plot(
    column="cluster",
    categorical=True,
    cmap=my_custom_cmap,
    legend=True,
)

In [ ]:
# choose number of clusters
subcluster_nums = [2, 2, 2, 2, 3, 2, 1]
sum(subcluster_nums)

In [ ]:
def run_subclustering(
    input_df: pd.DataFrame, subcluster_nums: list, num_clusters: int, n_init: int = 1
) -> pd.DataFrame:
    """
    Runs subclustering for each supergroup using KMeans and returns a modified DataFrame with subcluster labels.

    Parameters:
    - output_df (pd.DataFrame): The original DataFrame containing data and cluster assignments.
    - subcluster_nums (list): A list specifying the number of subclusters to split each supergroup into.
    - num_clusters (int): The total number of supergroups.
    - n_init (int, optional): The number of times KMeans will be initialized. Defaults to 1.

    Returns:
    - pd.DataFrame: A new the output dataFrame with an added 'subcluster' column.
    """

    cluster_labels = np.sort(input_df["cluster"].unique())
    print(f"Cluster labels found: {cluster_labels}")
    if len(subcluster_nums) != len(cluster_labels):
        raise ValueError(
            f"Length of subcluster_nums ({len(subcluster_nums)}) does not match num_clusters ({len(cluster_labels)})."
        )

    # Work on a copy of the DataFrame to prevent unintended modifications
    df = input_df.copy()

    for cluster, num_subclusters in zip(cluster_labels, subcluster_nums):
        print(f"Clustering supergroup {cluster} into {num_subclusters} subclusters.")

        # Select rows corresponding to the current cluster, drop the cluster column before clustering
        cluster_df = (
            input_df.query("cluster == @cluster").drop(columns="cluster").copy()
        )
        # Run KMeans clustering for the selected supergroup
        subcluster_output_df = cluster_df.copy()
        kmeans_sub = KMeans(
            n_clusters=num_subclusters,
            init="random",
            random_state=random_seed,
            n_init=n_init,
        )
        subcluster_output_df["cluster"] = kmeans_sub.fit_predict(cluster_df)

        # Combine names
        subcluster_output_df["subcluster"] = [
            str(cluster) + str(i) for i in subcluster_output_df["cluster"]
        ]

        # Update the modified DataFrame with subclustering results
        df.loc[cluster_df.index, "subcluster"] = subcluster_output_df["subcluster"]

    # Save the final output
    #  df[["cluster","subcluster"]].to_csv("outputs/subgroups_clusteroutput.csv")
    return df  # Return the modified DataFrame with clusters and subclusters


subgrouped_variable_df = run_subclustering(
    supergrouped_variable_df.drop(columns="geometry"),
    subcluster_nums,
    num_clusters=num_clusters,
    n_init=100,
)

In [ ]:
# number of OAs in each cluster
cluster_counts = subgrouped_variable_df["subcluster"].value_counts().sort_index()
# percentage of OAs in each cluster
cluster_perc = (cluster_counts / cluster_counts.sum() * 100).round(2)

# join pop_size to supergrouped_variable_df on index
subrgrouped_variable_df_withpop = subgrouped_variable_df.join(pop_size, how="left")

# pop in each cluster
cluster_pop = subrgrouped_variable_df_withpop.groupby("subcluster")[
    "Obyvatelstvo celkem"
].sum()
# percentage of pop in each cluster
cluster_pop_perc = (cluster_pop / cluster_pop.sum() * 100).round(2)

# combine into a dataframe
sub_cluster_summary = pd.DataFrame(
    {
        "Number of OAs": cluster_counts,
        "Percentage of OAs": cluster_perc,
        "Population": cluster_pop,
        "Percentage of Population": cluster_pop_perc,
    }
)
sub_cluster_summary

In [ ]:
# 1. Calculate percentage difference
global_means = subgrouped_variable_df[variables].mean()
subcluster_means = subgrouped_variable_df.groupby(["cluster", "subcluster"])[
    variables
].mean(numeric_only=True)

# Percentage difference: subcluster relative to global means
pct_diff_sub = (subcluster_means / global_means) * 100
pct_display_df_sub = pct_diff_sub.T

# 2. Flatten the MultiIndex columns using raw IDs
# This converts [(1, '1a'), (1, '1b')] into ["1-1a", "1-1b"]
pct_display_df_sub.columns = [f"{c[0]}-{c[1]}" for c in pct_display_df_sub.columns]

# 3. Build customdata for hover (handling ArrowStringArray if necessary)
feature_names = pct_display_df_sub.index.to_numpy(dtype=object)
customdata_pct_sub = np.tile(
    feature_names.reshape(-1, 1), (1, pct_display_df_sub.shape[1])
)

# 4. Heatmap
fig_pct_sub = px.imshow(
    pct_display_df_sub,
    color_continuous_scale="RdYlGn",
    origin="lower",
    aspect="auto",
    labels=dict(x="Subcluster", y="Feature", color="% of Global Mean"),
    zmin=0,
    zmax=200,
)

# Attach customdata and set hover
fig_pct_sub.data[0].customdata = customdata_pct_sub
fig_pct_sub.update_traces(
    hovertemplate="Subcluster: %{x}<br>Feature: %{customdata}<br>% of Global Mean: %{z:.1f}%<extra></extra>",
    zmid=100,
)

fig_pct_sub.update_layout(
    title="Subcluster Profiles (% of Global Mean)",
    xaxis_title="Subcluster (ClusterID-SubID)",
    yaxis_title="Feature",
    height=2500,
)

# 5. Add vertical lines at Cluster boundaries
# Now col.split("-")[0] will get the raw cluster ID string
col_clusters = [str(col).split("-")[0] for col in pct_display_df_sub.columns]

boundaries = [
    i + 0.5
    for i in range(len(col_clusters) - 1)
    if col_clusters[i] != col_clusters[i + 1]
]

for b in boundaries:
    fig_pct_sub.add_vline(x=b, line_width=2, line_dash="dash", line_color="black")

fig_pct_sub.show()

In [ ]:
subgrouped_variable_df["geometry"] = variable_df.geometry
subgrouped_variable_df = subgrouped_variable_df.set_geometry("geometry")

colors_groups = {
    # A: Blue (Base: #1f77b4)
    "A2": "#15527c",
    "C0": "#8c564b",
    "A0": "#1f77b4",
    # B: Orange (Base: #ff7f0e)
    "C1": "#17becf",
    "A1": "#ff7f0e",
    "B2": "#b3590a",
    # C: Green (Base: #2ca02c)
    "B0": "#98df8a",
    "B1": "#2ca02c",
    "C2": "#1e6e1e",
    # D: Purple/Pink (Base: #9467bd)
    "G1": "gray",
    "F0": "#dbbd24",
    "D2": "#674783",
    # E: Pink/Magenta (Base: #e377c2)
    "E0": "#d62728",
    "E1": "black",
    "E2": "#e377c2",
    "G0": "#f7b6d2",
    "D1": "#9467bd",
    "D0": "#c5b0d5",
    "F1": "#674783",
}
# 1. Get the unique subclusters actually present in your data
present_categories = sorted(subgrouped_variable_df["subcluster"].unique())

# 2. Create a list of colors only for those categories
shades_groups = [colors_groups[cat] for cat in present_categories]

# 3. Plot using this dynamically matched list
subgrouped_variable_df.plot(
    column="subcluster",
    categorical=True,
    cmap=ListedColormap(shades_groups),
    legend=True,
    legend_kwds={"bbox_to_anchor": (1, 1)},  # Keeps the legend tidy
)

In [ ]:
# interactive map
subgrouped_variable_df[["subcluster", "geometry"]].assign(
    geometry=lambda df: df.geometry.simplify(25, preserve_topology=True)
).explore(
    "subcluster",
    categorical=True,
    cmap=ListedColormap(shades_groups),
    legend=True,
    prefer_canvas=True,
)

In [ ]:
# 1. Setup dimensions and data
subclusters = pct_diff_sub.index.tolist()  # List of tuples like (1, 'A1')
categories = pct_diff_sub.columns.tolist()
num_subclusters = len(subclusters)

# Define grid layout
cols = 3
rows = (num_subclusters + cols - 1) // cols

fig_sub = make_subplots(
    rows=rows,
    cols=cols,
    specs=[[{"type": "polar"}] * cols] * rows,
    subplot_titles=[f"Subcluster {s[1]}" for s in subclusters],
)

# 2. Iterate and Plot
for i, (cluster_id, sub_id) in enumerate(subclusters):
    row = (i // cols) + 1
    col = (i % cols) + 1

    # --- COLOR LOGIC ---
    # Try to find the specific subcluster ID in your dictionary first
    # Fallback to a grey if not found
    specific_color = colors_groups.get(str(sub_id), "#7f7f7f")

    # Get values for this specific row in the MultiIndex
    r_values = list(pct_diff_sub.loc[(cluster_id, sub_id)].values)
    r_plot = r_values + [r_values[0]]
    theta_plot = categories + [categories[0]]

    fig_sub.add_trace(
        go.Scatterpolar(
            r=r_plot,
            theta=theta_plot,
            fill="toself",
            name=f"Sub {sub_id}",
            line=dict(color=specific_color, width=2),
            fillcolor=specific_color,
            opacity=0.6,
            mode="lines+markers",
            marker=dict(size=4),  # Small markers for readability
            hovertemplate="<b>Feature:</b> %{theta}<br><b>Value:</b> %{r:.1f}%<extra></extra>",
        ),
        row=row,
        col=col,
    )

# 3. Layout and Styling
fig_sub.update_layout(
    title_text="Subcluster Profiles (% of Global Mean)",
    height=400 * rows,
    showlegend=False,
    template="plotly_white",
    hovermode="closest",
)

fig_sub.update_polars(
    radialaxis=dict(
        visible=True,
        range=[0, 200],
        ticksuffix="%",
        gridcolor="rgba(0,0,0,0.1)",
        angle=45,  # Tilts the numbers so they don't overlap lines
    ),
    angularaxis=dict(
        showticklabels=False, ticks="", showline=True, linecolor="lightgrey"
    ),
)

fig_sub.show()

# Create sub-subgroups

In [ ]:
def run_sub_subclustering(
    input_df: pd.DataFrame, parent_col: str, branching_dict: dict, n_init: int = 10
) -> pd.DataFrame:
    """
    Splits groups into a deeper level based on a dictionary mapping.

    Parameters:
    - input_df: DataFrame with the current level of clustering.
    - parent_col: The column to split (e.g., 'subcluster').
    - branching_dict: Dict like {'A0': 3, 'A1': 2...}
    """
    df = input_df.copy()
    child_col = f"sub_{parent_col}"
    df[child_col] = None

    # We only iterate over the keys provided in the dictionary
    for parent_label, num_splits in branching_dict.items():
        # Filter for rows and drop non-numeric/label columns
        mask = df[parent_col] == parent_label
        features = (
            df[mask]
            .select_dtypes(include=[np.number])
            .drop(columns=[parent_col], errors="ignore")
        )

        if num_splits > 1:
            kmeans = KMeans(
                n_clusters=num_splits,
                init="random",
                random_state=random_seed,
                n_init=n_init,
            )
            labels = kmeans.fit_predict(features)
            # Create labels like 'A0.0', 'A0.1'
            df.loc[mask, child_col] = [f"{parent_label}.{i}" for i in labels]
        else:
            # If split is 1, just append .0 to maintain naming consistency
            df.loc[mask, child_col] = f"{parent_label}.0"

    return df

In [ ]:
def create_subcluster_grid_plots(df, level_column, n_init=10, min_k=1, max_k=10):
    """
    Generates clustergrams for every group in a 3-column grid layout.
    """
    label_cols = ["cluster", "subcluster", "sub_subcluster"]
    unique_labels = np.sort(df[level_column].unique())
    num_groups = len(unique_labels)

    # Calculate grid dimensions (3 columns, rows as needed)
    cols = 3
    rows = math.ceil(num_groups / cols)

    # Create the figure
    fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
    axes = axes.flatten()  # Flatten to iterate easily if it's a 2D array

    global_min, global_max = -100, 150

    for i, label in enumerate(unique_labels):
        group_df = (
            df[df[level_column] == label]
            .select_dtypes(include=[np.number])
            .drop(columns=label_cols, errors="ignore")
        )

        if len(group_df) <= max_k:
            axes[i].set_title(f"Skipped {label}: Too small")
            continue

        # Fit Clustergram
        cgram = Clustergram(
            range(min_k, max_k), n_init=n_init, random_state=random_seed, verbose=False
        )
        cgram.fit(group_df)

        # Plot into the specific subplot axis
        cgram.plot(ax=axes[i])
        axes[i].set_ylim(global_min, global_max)
        axes[i].set_title(f"Clustergram: {level_column} {label}")

    # Hide any unused subplot axes
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()


# Run it for your subclusters
create_subcluster_grid_plots(subgrouped_variable_df, level_column="subcluster")

In [ ]:
# Define your strategy
sub_sub_strategy = {
    "A0": 1,
    "A1": 1,
    "A2": 1,
    "B0": 1,
    "B1": 1,
    "C0": 2,
    "C1": 1,
    "D0": 1,
    "D1": 2,
    "D2": 1,  # d1-3
    "E0": 2,
    "E1": 2,
    "E2": 3,
}

# Run the clustering
sub_sub_df = run_sub_subclustering(
    input_df=subgrouped_variable_df,
    parent_col="subcluster",
    branching_dict=sub_sub_strategy,
    n_init=100,
)
(sorted(sub_sub_df["sub_subcluster"].unique()))

In [ ]:
# Based on your existing colors_expanded
sub_sub_colors = {
    # Cluster A variations
    "A0.0": "#1f77b4",
    "A0.1": "#6ab0e2",
    "A1.0": "#8c564b",
    "A1.1": "#c49c94",
    "A2.0": "#15527c",
    "A2.1": "#4a90c0",
    # Cluster B variations
    "B0.0": "#17becf",
    "B0.1": "#9edae5",
    "B1.0": "#ff7f0e",
    "B1.1": "#ffbb78",
    # Cluster C variations
    "C0.1": "#2ca02c",
    "C0.0": "#185200",
    "C1.0": "#98df8a",
    "C1.1": "#98df8a",
    # Cluster D variations
    "D0.0": "#c5b0d5",
    "D0.1": "#e7d4f8",
    "D1.0": "#9467bd",
    "D1.1": "#dbbd24",
    "D2.0": "#674783",
    "D2.1": "#a084bd",
    # Cluster E variations
    "E0.0": "#d62728",
    "E0.1": "#ffaf94",
    "E0.2": "#33dba9",
    "E1.0": "#e377c2",
    "E1.1": "#000000",
    "E2.0": "#f7b6d2",
    "E2.1": "#94b814",
    "E2.2": "khaki",
}

In [ ]:
# 1. Get the unique subclusters actually present in your data
present_categories = sorted(sub_sub_df["sub_subcluster"].unique())

# 2. Create a list of colors only for those categories
shades_to_use = [sub_sub_colors[cat] for cat in present_categories]

# 3. Plot using this dynamically matched list
sub_sub_df.plot(
    column="sub_subcluster",
    categorical=True,
    cmap=ListedColormap(shades_to_use),
    legend=True,
    legend_kwds={"bbox_to_anchor": (1, 1)},  # Keeps the legend tidy
)

In [ ]:
# 1. Join population data to your sub_sub_df
sub_sub_df_withpop = sub_sub_df.join(pop_size, how="left")

# 2. Calculate counts and percentages for OAs
ss_counts = sub_sub_df_withpop["sub_subcluster"].value_counts().sort_index()
ss_perc = (ss_counts / ss_counts.sum() * 100).round(2)

# 3. Calculate Population totals and percentages
ss_pop = sub_sub_df_withpop.groupby("sub_subcluster")["Obyvatelstvo celkem"].sum()
ss_pop_perc = (ss_pop / ss_pop.sum() * 100).round(2)

# 4. Create the summary DataFrame
sub_sub_summary = pd.DataFrame(
    {
        "Number of OAs": ss_counts,
        "Percentage of OAs": ss_perc,
        "Population": ss_pop,
        "Percentage of Population": ss_pop_perc,
    }
)

# 5. Optional: Add the Parent Subcluster for easier reading
sub_sub_summary["Parent Subcluster"] = sub_sub_summary.index.str.split(".").str[0]

# Reorder columns to put Parent first for clarity
sub_sub_summary = sub_sub_summary[
    [
        "Parent Subcluster",
        "Number of OAs",
        "Percentage of OAs",
        "Population",
        "Percentage of Population",
    ]
]

sub_sub_summary

In [ ]:
# interactive map
sub_sub_df[["sub_subcluster", "geometry"]].assign(
    geometry=lambda df: df.geometry.simplify(25, preserve_topology=True)
).explore(
    "sub_subcluster",
    categorical=True,
    cmap=ListedColormap(shades_to_use),
    legend=True,
    prefer_canvas=True,
)